In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")


In [0]:
catalogo = dbutils.widgets.get("catalogo")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

In [0]:
orders_df = spark.table(f"{catalogo}.{bronze_schema}.orders_raw")
customers_df = spark.table(f"{catalogo}.{bronze_schema}.customers_raw")
order_items_df = spark.table(f"{catalogo}.{bronze_schema}.order_items_raw")
products_df = spark.table(f"{catalogo}.{bronze_schema}.products_raw")
payments_df = spark.table(f"{catalogo}.{bronze_schema}.payments_raw")
category_translation_df = spark.table(f"{catalogo}.{bronze_schema}.category_translation_raw")

In [0]:
orders_clean_df = orders_df.select(
    col("order_id"),
    col("customer_id"),
    lower(trim(col("order_status"))).alias("order_status"),
    col("order_purchase_timestamp"),
    col("order_approved_at"),
    col("order_delivered_carrier_date"),
    col("order_delivered_customer_date"),
    col("order_estimated_delivery_date")
).dropDuplicates(["order_id"])

In [0]:
customers_clean_df = customers_df.select(
    col("customer_id"),
    col("customer_unique_id"),
    lower(trim(col("customer_city"))).alias("customer_city"),
    upper(trim(col("customer_state"))).alias("customer_state")
).dropDuplicates(["customer_id"])

In [0]:
order_items_clean_df = order_items_df.select(
    col("order_id"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date"),
    col("price"),
    col("freight_value")
).filter(
    (col("order_id").isNotNull()) &
    (col("order_item_id").isNotNull()) &
    (col("price") >= 0) &
    (col("freight_value") >= 0)
).dropDuplicates(["order_id", "order_item_id"])

In [0]:
category_translation_clean_df = category_translation_df.select(
    lower(trim(col("product_category_name"))).alias("product_category_name"),
    lower(trim(col("product_category_name_english"))).alias("product_category_name_english")
).dropDuplicates(["product_category_name"])

In [0]:
products_clean_df = products_df.select(
    col("product_id"),
    lower(trim(col("product_category_name"))).alias("product_category_name"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
).withColumn(
    "product_volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
).dropDuplicates(["product_id"])

In [0]:
products_enriched_df = products_clean_df.join(
    category_translation_clean_df,
    on="product_category_name",
    how="left"
)

In [0]:
payments_clean_df = payments_df.select(
    col("order_id"),
    lower(trim(col("payment_type"))).alias("payment_type"),
    col("payment_installments"),
    col("payment_value")
).filter(
    (col("order_id").isNotNull()) &
    (col("payment_value") >= 0)
)

In [0]:
payments_by_order_df = payments_clean_df.groupBy("order_id").agg(
    first("payment_type").alias("payment_type"),
    max("payment_installments").alias("payment_installments"),
    sum("payment_value").alias("payment_value")
)

In [0]:

def get_customer_region(state):
    if state in ["SP", "RJ", "MG", "ES"]:
        return "Southeast"
    elif state in ["PR", "SC", "RS"]:
        return "South"
    elif state in ["DF", "GO", "MT", "MS"]:
        return "Central-West"
    elif state in ["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"]:
        return "Northeast"
    elif state in ["AM", "RR", "AP", "PA", "TO", "RO", "AC"]:
        return "North"
    else:
        return "Unknown"

customer_region_udf = udf(get_customer_region, StringType())

In [0]:
silver_df = orders_clean_df \
    .join(customers_clean_df, on="customer_id", how="left") \
    .join(order_items_clean_df, on="order_id", how="left") \
    .join(products_enriched_df, on="product_id", how="left") \
    .join(payments_by_order_df, on="order_id", how="left")

In [0]:
silver_final_df = silver_df.select(
    col("order_id"),
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_city"),
    col("customer_state"),
    customer_region_udf(col("customer_state")).alias("customer_region"),
    col("order_status"),
    col("order_purchase_timestamp"),
    to_date(col("order_purchase_timestamp")).alias("order_purchase_date"),
    col("order_approved_at"),
    col("order_delivered_carrier_date"),
    col("order_delivered_customer_date"),
    col("order_estimated_delivery_date"),
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")).alias("delivery_days"),
    when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), True)
        .otherwise(False).alias("is_late_delivery"),
    when(col("order_delivered_customer_date").isNull(), "Not Delivered")
        .when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "Late")
        .otherwise("On Time").alias("delivery_status"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date"),
    col("price"),
    col("freight_value"),
    col("product_category_name"),
    col("product_category_name_english"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm"),
    col("product_volume_cm3"),
    col("payment_type"),
    col("payment_installments"),
    col("payment_value"),
    current_timestamp().alias("transformation_timestamp")
)

In [0]:
silver_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{silver_schema}.olist_orders_enriched")

In [0]:
display(silver_final_df.limit(10))